# R3_03 — Pilihan berbasis DEV dan pembekuan (frozen_choices.json)

Dijalankan setelah ke-9 run R3_02 selesai. **Tidak memuat data uji** (penjaga aktif).

1. Cek kelengkapan 9 run (3 model × seed 42/123/7) dan konfigurasi identik; cek sha256 checkpoint & prediksi dev.
2. Agregasi probabilitas dev antar-seed (`aggregate_seeds` NB03).
3. Bobot weighted soft voting = macro-F1 dev per aspek (sel NB05).
4. Model individu terbaik = Overall macro-F1 dev tertinggi (seri → ASC F1 dev → IndoBERT, XLM-R, mBERT) — rencana §5.
5. Grid ablasi ensemble Tabel 17 (sel NB05B, dihitung di dev); skema terpilih = ASC F1 dev tertinggi (seri → Overall F1 dev → paling
   sederhana) — rencana §6. Rekonstruksi tiap skema dari kuncinya diverifikasi terhadap hasil sel NB05B.
6. Ensemble utama di dev (sel NB05) dan jumlah kasus yang akan dirujuk ke LLM di dev (sel NB06 `needs_llm_judge`).
7. Inferensi ablasi V1–V6 di dev (checkpoint baru) dan tabel disagreement di dev (sel NB04).
8. Tulis `outputs_round3/frozen_choices.json` (semua pilihan + sha256 checkpoint, data, dan kode R3_04–R3_07).

Waktu: ±5–10 menit (inferensi ablasi 54 kali × 240 pasangan di GPU).

In [ ]:
# ── PARAMETER ──
ALLOW_SMOKE_RUNS = False         # True hanya untuk uji pengembang (run smoke test)
RUN_ABLATION_INFERENCE = True    # inferensi V1–V6 di dev
FORCE_REFREEZE = False           # True = menimpa frozen_choices.json yang sudah ada (catat alasannya di log rencana!)

import os
ALLOW_SMOKE_RUNS = os.environ.get('R3_ALLOW_SMOKE', str(ALLOW_SMOKE_RUNS)) == 'True'
FORCE_REFREEZE = os.environ.get('R3_FORCE_REFREEZE', str(FORCE_REFREEZE)) == 'True'

In [ ]:
import sys, json, shutil
sys.path.insert(0, os.environ.get('R3_CODE_DIR', os.getcwd()))
import numpy as np, pandas as pd
import config as C
import r3_lib as L
L.forbid_test_access()
import torch
RUNS_ROOT = (C.OUT_R3 / 'smoke_runs') if ALLOW_SMOKE_RUNS else C.RUNS_DIR
DEV_PRED = C.DEV_DIR / 'predictions'
DEV_PRED.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Run dari:', RUNS_ROOT, '| output:', C.DEV_DIR, '| device:', DEVICE)
if ALLOW_SMOKE_RUNS:
    print('⚠️  MODE UJI: memakai run smoke test; frozen_choices.json ditandai smoke=True dan ditolak R3_04 mode nyata')

## 1. Kelengkapan dan kesamaan konfigurasi 9 run

In [ ]:
r01 = json.loads((C.DATA_R3 / 'R3_01_manifest.json').read_text(encoding='utf-8'))
dev_sha = L.sha256_file(C.DEV_PAIRS)
rows, fps, runs = [], [], {}
for m in C.MODELS:
    for s in C.SEEDS:
        rd = RUNS_ROOT / f'{m}_seed{s}'
        info_p = rd / 'run_info.json'
        assert (rd / 'DONE').exists() and info_p.exists(), f'Run belum selesai: {rd}'
        info = json.loads(info_p.read_text(encoding='utf-8'))
        assert info['status'] == 'complete', rd
        if info.get('smoke') and not ALLOW_SMOKE_RUNS:
            raise RuntimeError(f'{rd} adalah smoke test — tidak boleh dipakai')
        fp = info['fingerprint']
        assert fp['train_sha256'] == r01['outputs']['train_pairs.csv'], f'{rd}: train_pairs ≠ R3_01'
        assert fp['dev_sha256'] == dev_sha, f'{rd}: dev_pairs berbeda'
        wfile = next(rd / 'best' / f for f in ('model.safetensors', 'pytorch_model.bin') if (rd / 'best' / f).exists())
        dfile = rd / f'{m}_seed{s}_dev.csv'
        msha, dsha = L.sha256_file(wfile), L.sha256_file(dfile)
        assert msha == info['best_model_sha256'] and dsha == info['dev_predictions_sha256'], f'{rd}: file berubah setelah pelatihan'
        fps.append({k: v for k, v in fp.items() if k not in ('model_name', 'seed', 'model_id')})
        runs[f'{m}_seed{s}'] = {'dir': str(rd), 'model_id': fp['model_id'], 'best_model_path': str(wfile), 'best_model_sha256': msha,
                                'dev_predictions_sha256': dsha, 'best_epoch': info['best_epoch'], 'best_dev_f1': info['best_dev_f1'],
                                'total_train_seconds': info['total_train_seconds'], 'gpu': info['environment'].get('gpu'),
                                'torch': info['environment'].get('torch'), 'transformers': info['environment'].get('transformers'),
                                'smoke': bool(info.get('smoke'))}
        shutil.copy(dfile, DEV_PRED / f'{m}_seed{s}_dev.csv')
assert all(f == fps[0] for f in fps), 'Konfigurasi 9 run tidak identik'
print('✓ 9 run lengkap; konfigurasi identik; checkpoint & prediksi dev cocok dengan run_info.json')
pd.DataFrame(runs).T[['best_epoch', 'best_dev_f1', 'total_train_seconds', 'gpu', 'torch', 'transformers']]

## 2–4. Agregasi seed, bobot soft voting, model individu terbaik

In [ ]:
# Sel asli NB03 juga menjalankan loop agregasi (split dev & test) di PRED_DIR; di sini PRED_DIR = folder prediksi dev
# (tidak ada file uji di sana → split test hanya mencetak 'tidak ditemukan').
agg_ns = L.exec_legacy('nb03_aggregate_seeds', L.base_namespace(PRED_DIR=str(DEV_PRED), SEEDS=C.SEEDS))
dev_preds = {}
for m in C.MODELS:
    d = agg_ns['aggregate_seeds'](m, 'dev', C.SEEDS, str(DEV_PRED))
    d.to_csv(DEV_PRED / f'{m}_dev_avg.csv', index=False, encoding='utf-8-sig')
    d['pair_id'] = d['pair_id'].astype(str)
    dev_preds[m] = d

df_dev = pd.read_csv(C.DEV_PAIRS, dtype={'tweet_id': str}); df_dev['pair_id'] = df_dev['pair_id'].astype(str)
df_dev['aspek'] = df_dev['pair_id'].map(lambda p: next(a for a in C.ASPECTS if p.endswith('_' + a)))

w_ns = L.exec_legacy('nb05_weights', L.base_namespace(df_dev=df_dev, dev_preds=dev_preds))
model_weights = w_ns['model_weights']

h = L.exec_legacy('nb05b_helpers', L.base_namespace(test_preds=dev_preds, dev_preds=dev_preds, df_test=df_dev, df_dev=df_dev))
y_dev = df_dev['label'].values
indiv = {}
for m in C.MODELS:
    pred = df_dev[['pair_id']].merge(dev_preds[m][['pair_id', 'pred_label']], on='pair_id')['pred_label'].values
    indiv[m] = h['compute_metrics'](y_dev, pred)
order = list(C.MODELS)
best_individual = sorted(order, key=lambda m: (-round(indiv[m]['overall'], 12), -round(indiv[m]['asc'], 12), order.index(m)))[0]
print('Model individu (dev):', {m: {k: round(v, 4) for k, v in r.items()} for m, r in indiv.items()})
print('→ model individu terbaik (dev):', best_individual)

## 5. Grid ablasi ensemble Tabel 17 — dihitung di DEV (sel NB05B apa adanya)

In [ ]:
L.exec_legacy(['nb05b_weights', 'nb05b_diag', 'nb05b_exp0_baselines', 'nb05b_exp1_subsets', 'nb05b_exp2_power',
               'nb05b_exp3_temperature', 'nb05b_exp4_combo'], h)
all_results, best_T, mw_dev = h['all_results'], h['best_T'], h['model_weights_dev']
assert mw_dev == model_weights, 'bobot NB05B ≠ NB05'
selected = L.select_scheme(all_results)
# Verifikasi: rekonstruksi tiap skema dari kuncinya = hasil sel NB05B
for k, r in all_results.items():
    pred = L.scheme_predict(k, h, h['probs_all'], mw_dev, df_dev['aspek'], best_T)
    rr = h['compute_metrics'](y_dev, pred)
    assert all(abs(rr[x] - r[x]) < 1e-12 for x in ('acd', 'asc', 'overall')), k
grid = (pd.DataFrame(all_results).T.rename(columns={'acd': 'ACD_dev', 'asc': 'ASC_dev', 'overall': 'Overall_dev'})
        .assign(kompleksitas=lambda d: [L.scheme_complexity(k) for k in d.index]).sort_values('ASC_dev', ascending=False))
grid.to_csv(C.DEV_DIR / 'table17_grid_dev.csv', encoding='utf-8-sig')
print(f'{len(all_results)} skema; T terbaik (dev) = {best_T}; skema terpilih: {selected} '
      f"(ASC dev {all_results[selected]['asc']:.4f}, Overall dev {all_results[selected]['overall']:.4f})")
print('✓ scheme_predict mereproduksi metrik dev semua skema NB05B')
grid.head(10).round(4)

## 6. Ensemble utama di dev (sel NB05) dan kasus yang akan dirujuk ke LLM

In [ ]:
e_ns = L.exec_legacy(['nb05_vectorized_ensemble', 'nb05_run_ensemble'],
                     L.base_namespace(test_preds=dev_preds, df_test=df_dev, model_weights=model_weights, DEVICE=DEVICE))
res = e_ns['results']
p_ns = L.exec_legacy('nb06_prompt', L.base_namespace())
ens_dev = pd.DataFrame({'pair_id': df_dev.pair_id, 'aspek': df_dev.aspek, 'true_label': df_dev.label,
                        'hard_vote_label': res['hard_labels'], 'vote_type': np.where(res['vote_type_bulat'], 'bulat', 'terpecah'),
                        'soft_vote_label': res['soft_labels'], 'soft_vote_margin': np.round(res['margins'], 6),
                        **{f'pred_{m}': res['preds_per_model'][m] for m in C.MODELS}})
ens_dev['needs_llm'] = [p_ns['needs_llm_judge'](v, mg, C.MARGIN_THRESHOLD) for v, mg in zip(ens_dev.vote_type, ens_dev.soft_vote_margin)]
ens_dev.to_csv(C.DEV_DIR / 'ensemble_dev.csv', index=False, encoding='utf-8-sig')
dev_main = {name: h['compute_metrics'](y_dev, ens_dev[col].values) for name, col in
            [('Hard Voting', 'hard_vote_label'), ('Weighted Soft Voting', 'soft_vote_label')]}
print('Ensemble dev:', {k: {x: round(v, 4) for x, v in r.items()} for k, r in dev_main.items()})
print(f"Kasus dev yang akan dirujuk ke LLM: {int(ens_dev.needs_llm.sum())} dari {len(ens_dev)}")

## 7. Disagreement (sel NB04) dan ablasi preprocessing V1–V6 di dev

In [ ]:
d_ns = L.exec_legacy(['nb04_analysis_frame', 'nb04_disagreement'], L.base_namespace(df_dev=df_dev, preds_dev=dev_preds))
disagr = d_ns['disagr_matrix'] * 100
per_asp = pd.DataFrame({a: {f'{m1} vs {m2}': 100 * d_ns['disagreement_rate'](d_ns['df_analysis'][d_ns['df_analysis'].aspek == a],
                                                                              f'pred_{m1}', f'pred_{m2}')
                            for i, m1 in enumerate(C.MODELS) for m2 in list(C.MODELS)[i + 1:]} for a in C.ASPECTS})
disagr.to_csv(C.DEV_DIR / 'disagreement_dev.csv', encoding='utf-8-sig')
per_asp.to_csv(C.DEV_DIR / 'disagreement_dev_per_aspect.csv', encoding='utf-8-sig')
disagr.round(1)

In [ ]:
ablation_dev = None
if RUN_ABLATION_INFERENCE:
    import gc
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    t_ns = L.exec_legacy(['nb03_dataset', 'nb03_train_functions'], L.base_namespace())
    texts = pd.read_csv(C.TWEETS_PREPROCESSED, usecols=['tweet_id'] + [f'text_{v}' for v in C.TEXT_VERSIONS_ABLATION], dtype={'tweet_id': str})
    dv = df_dev.merge(texts, on='tweet_id', how='left')
    assert len(dv) == len(df_dev) and (dv.text_v1.astype(str) == dv.sentence_a.astype(str)).all(), 'text_v1 ≠ sentence_a dev'
    crit = torch.nn.CrossEntropyLoss()
    probs = {}
    for m in C.MODELS:
        for s in C.SEEDS:
            bd = os.path.dirname(runs[f'{m}_seed{s}']['best_model_path'])
            tok = AutoTokenizer.from_pretrained(bd)
            mdl = AutoModelForSequenceClassification.from_pretrained(bd, num_labels=C.NUM_LABELS, ignore_mismatched_sizes=True).to(DEVICE)
            for v in C.TEXT_VERSIONS_ABLATION:
                dfv = dv[['pair_id', 'sentence_b']].assign(sentence_a=dv[f'text_{v}'].fillna('').astype(str))
                loader = torch.utils.data.DataLoader(t_ns['SentencePairDataset'](dfv, tok, C.MAX_LENGTH), batch_size=C.BATCH_SIZE, shuffle=False)
                _, _, _, pr, ids, _ = t_ns['evaluate'](mdl, loader, crit, DEVICE)
                assert list(ids) == list(dfv.pair_id)
                probs.setdefault((m, v), []).append(np.array(pr))
            del mdl, tok; gc.collect()
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
    rows = []
    (C.DEV_DIR / 'ablation').mkdir(exist_ok=True)
    for (m, v), lst in probs.items():
        avg = np.mean(lst, axis=0); pred = avg.argmax(1)
        pd.DataFrame({'pair_id': dv.pair_id, **{c: avg[:, i] for i, c in enumerate(C.PROB_COLS)}, 'pred_label': pred}).to_csv(
            C.DEV_DIR / 'ablation' / f'{m}_{v}_dev_avg.csv', index=False, encoding='utf-8-sig')
        rr = h['compute_metrics'](y_dev, pred)
        rows.append({'model': m, 'versi': v.upper(), 'ACD_dev': rr['acd'], 'ASC_dev': rr['asc'], 'Overall_dev': rr['overall']})
        if v == 'v1':   # V1 harus sama dengan prediksi dev dari pelatihan
            ref = df_dev[['pair_id']].merge(dev_preds[m], on='pair_id')
            agree = float((ref.pred_label.values == pred).mean()); dmax = float(np.abs(ref[C.PROB_COLS].values - avg).max())
            print(f'  {m}: V1 vs prediksi dev pelatihan — label sama {agree:.1%}, selisih prob maks {dmax:.2e}')
            assert agree > 0.99
    ablation_dev = pd.DataFrame(rows)
    ablation_dev.to_csv(C.DEV_DIR / 'ablation_v1_v6_dev.csv', index=False, encoding='utf-8-sig')
    display(ablation_dev.pivot(index='versi', columns='model', values='Overall_dev').round(4))

## 8. Bekukan pilihan → frozen_choices.json

In [ ]:
if C.FROZEN_JSON.exists() and not FORCE_REFREEZE:
    raise RuntimeError(f'{C.FROZEN_JSON} sudah ada. Pilihan sudah dibekukan; set FORCE_REFREEZE=True hanya bila ada alasan '
                       'yang dicatat di Log perubahan rencana_latih_ulang.md (dan SEBELUM data uji disentuh).')
missing_code = [nb for nb in L.FROZEN_CODE_NOTEBOOKS if not (C.CODE_DIR / nb).exists()]
assert not missing_code, f'Notebook evaluasi belum ada: {missing_code}'
frozen = {
    'created_at': L.now_utc(), 'smoke': ALLOW_SMOKE_RUNS, 'runs_root': str(RUNS_ROOT),
    'plan_sha256': L.sha256_file(C.BASE_DIR / 'revisi_round3' / 'rencana_latih_ulang.md'),
    'config': C.frozen_config_dict(),
    'runs': runs,
    'dev_avg_sha256': {m: L.sha256_file(DEV_PRED / f'{m}_dev_avg.csv') for m in C.MODELS},
    'model_weights': model_weights,
    'best_individual': {'model': best_individual, 'rule': 'Overall F1 dev → ASC F1 dev → urutan IndoBERT, XLM-R, mBERT',
                        'dev_metrics': indiv},
    'table17': {'selected_key': selected, 'rule': 'ASC F1 dev → Overall F1 dev → paling sederhana → kunci',
                'dev_metrics': all_results[selected], 'best_T': best_T, 'model_weights_dev': mw_dev,
                'n_schemes': len(all_results), 'grid_file': str(C.DEV_DIR / 'table17_grid_dev.csv')},
    'dev_reference': {'ensemble': dev_main, 'n_flagged_dev': int(ens_dev.needs_llm.sum()),
                      'disagreement_pct': disagr.round(4).to_dict()},
    'ablation_dev_file': str(C.DEV_DIR / 'ablation_v1_v6_dev.csv') if ablation_dev is not None else None,
    'data_sha256': L.data_fingerprints(),
    'code_sha256': L.code_fingerprints(),
    'environment': L.environment_info(),
}
L.write_json(C.FROZEN_JSON, frozen)
print(f'✓ Dibekukan: {C.FROZEN_JSON}')
print(json.dumps({'best_individual': best_individual, 'table17': selected, 'best_T': best_T, 'n_flagged_dev': frozen['dev_reference']['n_flagged_dev']}, indent=2))